# Simulaciones:  variabilidad y fiabilidad del anotador LLM

Los LLM son probabilísticos: el mismo texto de un padre puede dar anotaciones distintas en ejecuciones distintas. Para un uso clínico eso es un problema de fiabilidad de la medida. En este cuaderno mido esa variabilidad y miro qué parámetros la estabilizan.

Es un experimento factorial:

- Variables independientes (los parámetros que muevo): backend (directo / langchain / tool_calling / instructor), modelo, temperatura, top_p, seed, estrategia de prompt.
- Réplicas: N repeticiones por (entrada × configuración), que son las que me dejan medir la varianza.
- Variables dependientes (las métricas):
  - Consistencia entre repeticiones (Jaccard de ítems/escalas, acuerdo modal del nivel).
  - Fiabilidad entre réplicas: alpha de Krippendorff del nivel de alerta.
  - Calidad de la salida: las 5 métricas sin ground truth.
  - Coste: latencia y cuántas veces falla el formato.

Todo se guarda en la tabla `anotacion` de la base de datos.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from anotador.instrumento import cargar_instrumento
from anotador.simulacion import Rejilla, ejecutar
from anotador.analisis import (
    cargar_df,
    consistencia_por_grupo,
    krippendorff_nivel,
    resumen_por_parametro,
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

instrumento = cargar_instrumento()
print(f"Instrumento: {instrumento['nombre']} ({len(instrumento['items'])} ítems)")

## 1  Definir la rejilla

Cada lista es un eje del experimento. El número de llamadas al modelo es `configuraciones × repeticiones × entradas`, así que conviene estimarlo antes de lanzar.


In [ ]:
rejilla = Rejilla(
    entradas=[1, 2],                 # None = todas las entradas de la BD
    backends=["langchain"],          # ["directo", "langchain", "tool_calling", "instructor"]
    modelos=["gemma4:e4b"],          # modelo pequeño para iterar rápido
    temperaturas=[0.0, 0.8],         # la temperatura es lo que más mueve la aleatoriedad
    top_ps=[None],
    seeds=[None],                    # None = seed aleatoria por repetición
    estrategias=["zero_shot"],
    repeticiones=3,                  # réplicas por (entrada × config)
)

n_entradas = len(rejilla.entradas or [])
print(f"Configuraciones: {len(rejilla.configuraciones())}")
print(f"Llamadas totales al modelo: {rejilla.total_ejecuciones(n_entradas)}")

In [ ]:
# Lanza la simulación (cada resultado se persiste en la tabla `anotacion`).
resultados = ejecutar(rejilla, instrumento, verbose=True)

## 2  Cargar todas las anotaciones

`cargar_df()` lee la tabla `anotacion` entera, no solo la última simulación. Así los resultados se van acumulando entre ejecuciones, que es cómodo para ir montando el experimento por partes.

In [ ]:
df = cargar_df()
print(f"{len(df)} anotaciones en la base de datos")
df[["id_entrada", "backend", "modelo", "temperature", "repeticion",
    "items_detectados", "nivel_alerta", "n_metricas_ok", "latencia_s"]].head(12)

## 3 Consistencia entre repeticiones

Para cada (entrada × configuración), cuánto se parecen las N repeticiones entre sí:

- `jaccard_items` / `jaccard_escalas` van de 0 a 1: 1.0 quiere decir que el modelo elige siempre lo mismo.
- `acuerdo_nivel` (de 0 a 1): qué fracción de repeticiones coincide con el nivel más frecuente.

Si una configuración da 1.0, es reproducible. Si da poco, manda el azar.

In [ ]:
cons = consistencia_por_grupo(df)
cons[["id_entrada", "backend", "temperature", "n_rep",
      "jaccard_items", "jaccard_escalas", "acuerdo_nivel", "media_metricas"]]

## 4 Fiabilidad entre réplicas: alpha de Krippendorff

Alpha del nivel de alerta (ordinal: bajo < moderado < alto), tratando cada repetición como un codificador y cada entrada como una unidad. La interpretación habitual: alpha ≥ 0.80 fiable, entre 0.67 y 0.80 aceptable, por debajo de 0.67 insuficiente.

Hace falta tener ≥ 2 repeticiones y varias entradas para que diga algo. Con datos sin variabilidad (por ejemplo temperatura 0, que da siempre lo mismo) el alpha sale indefinido (`NaN`), y eso ya indica que es determinista. Para un alpha robusto hay que subir el número de entradas.

In [ ]:
krippendorff_nivel(df)

## 5 — Efecto de cada parámetro

Resumen agrupado por un parámetro (aquí, la temperatura): consistencia, calidad media, tasa de fallo de formato y latencia. Cambia `parametro` por `"modelo"`, `"backend"`, etc.

In [ ]:
parametro = "temperature"
resumen = resumen_por_parametro(df, parametro)
display(resumen)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(resumen[parametro], resumen["jaccard_items"], "o-", label="Jaccard ítems")
ax[0].plot(resumen[parametro], resumen["acuerdo_nivel"], "s-", label="Acuerdo nivel")
ax[0].set_xlabel(parametro); ax[0].set_ylabel("consistencia"); ax[0].set_ylim(0, 1.05)
ax[0].set_title("Consistencia vs parámetro"); ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].plot(resumen[parametro], resumen["media_latencia"], "d-", color="tab:red")
ax[1].set_xlabel(parametro); ax[1].set_ylabel("latencia media (s)")
ax[1].set_title("Coste vs parámetro"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6 De las simulaciones a los hallazgos

Ideas para la tesis:

1. ¿Cuánta variabilidad hay? Mirar `jaccard_items` y `acuerdo_nivel` con temperatura > 0. Si caen mucho, el sistema no es reproducible sin hacer algo más.
2. ¿Qué reduce esa variabilidad? Comparar configuraciones: temperatura 0, salida estructurada (langchain / instructor), modelos más grandes.
3. Mitigación con self-consistency: ejecutar N veces y quedarse con la anotación de consenso (voto mayoritario por ítem). Mejora la fiabilidad pero gasta N veces más cómputo.
4. El compromiso entre fiabilidad y coste: cruzar `alpha_nivel` o la consistencia con `media_latencia`.

Lo que suele pasar: con temperatura 0 la consistencia es casi 1.0 (reproducible) pero el modelo puede perder matices; al subir la temperatura aumenta la variabilidad. La combinación de salida estructurada más voto mayoritario sobre N repeticiones suele dar el mejor equilibrio.

Cosas que quiero ampliar:
- Incluir más entradas (pacientes) para tener un alpha de Krippendorff más sólido. (En prueba con 30 pacientes, dos dias de proceso)
- Probar los backends y varios modelos (de e2b a 31b).
- Dejar el consenso por voto mayoritario como una estrategia más.

Para más adelante (no ahora): la tabla `referencia_sintetica` guarda la anotación del dataset sintético. No es un ground truth clínico validado, así que no la uso para evaluar la validez del sistema en esta fase. Eso lo dejo para cuando tenga anotaciones reales de clínicos.

## 7 Mitigación: voto mayoritario (self-consistency)

La idea para reducir la variabilidad: en vez de fiarme de una sola ejecución, lanzo el caso `n_muestras` veces y construyo una anotación de consenso (un ítem entra si lo eligen al menos el 50 % de las ejecuciones; el nivel se decide por mayoría).

Aquí comparo la anotación individual con el consenso para varios tamaños de lote `n_muestras` y dibujo la curva de consistencia frente a coste.

Coste: el consenso multiplica las llamadas por `n_muestras`. Conviene empezar con pocos casos y lotes pequeños. La celda de abajo lo estima antes de lanzar.

In [ ]:
# Parámetros de la comparación (ajusta según el tiempo disponible)
ENTRADAS_SC = [1, 2]        # casos a evaluar
REPETICIONES_SC = 3         # meta-réplicas por configuración (para medir su consistencia)
TEMP_SC = 0.7               # temperatura alta: ahí la variabilidad importa
NS_MUESTRAS = [3, 5]        # tamaños de lote del consenso a probar
MODELO_SC = "gemma4:e4b"
BACKEND_SC = "langchain"

# Estimación de coste (llamadas reales al modelo)
coste_individual = 1 * REPETICIONES_SC * len(ENTRADAS_SC)
coste_consenso = sum(n * REPETICIONES_SC * len(ENTRADAS_SC) for n in NS_MUESTRAS)
print(f"Llamadas al modelo estimadas: individual={coste_individual} + "
      f"consenso={coste_consenso} = {coste_individual + coste_consenso}")

In [ ]:
# 1) Línea base individual
ejecutar(
    Rejilla(entradas=ENTRADAS_SC, backends=[BACKEND_SC], modelos=[MODELO_SC],
            temperaturas=[TEMP_SC], repeticiones=REPETICIONES_SC,
            agregaciones=["individual"]),
    instrumento, verbose=False,
)

# 2) Consenso para cada tamaño de lote
for n in NS_MUESTRAS:
    ejecutar(
        Rejilla(entradas=ENTRADAS_SC, backends=[BACKEND_SC], modelos=[MODELO_SC],
                temperaturas=[TEMP_SC], repeticiones=REPETICIONES_SC,
                agregaciones=["consenso_voto"], n_muestras=n),
        instrumento, verbose=False,
    )
    print(f"  consenso n_muestras={n} ✓")
print("Comparación completada.")

In [ ]:
# Curva consistencia ↔ coste según n_muestras.
# Al agrupar por n_muestras, el bucket n=1 es la línea base individual.
df = cargar_df()
curva = resumen_por_parametro(df, "n_muestras").sort_values("n_muestras")
display(curva[["n_muestras", "jaccard_items", "acuerdo_nivel",
               "media_metricas", "media_latencia"]])

fig, ax1 = plt.subplots(figsize=(7, 4.5))
ax1.plot(curva["n_muestras"], curva["jaccard_items"], "o-", color="tab:blue",
         label="Consistencia (Jaccard ítems)")
ax1.plot(curva["n_muestras"], curva["acuerdo_nivel"], "s--", color="tab:green",
         label="Acuerdo nivel")
ax1.set_xlabel("n_muestras  (1 = individual)")
ax1.set_ylabel("consistencia"); ax1.set_ylim(0, 1.05); ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(curva["n_muestras"], curva["media_latencia"], "d-", color="tab:red",
         label="Latencia media (s)")
ax2.set_ylabel("latencia media (s)", color="tab:red")

l1, lab1 = ax1.get_legend_handles_labels()
l2, lab2 = ax2.get_legend_handles_labels()
ax1.legend(l1 + l2, lab1 + lab2, loc="lower right")
plt.title("Voto mayoritario: consistencia y coste vs nº de muestras")
plt.tight_layout(); plt.show()

### Lectura del resultado

- `n_muestras = 1` es la anotación individual, que a temperatura alta suele ser la menos consistente.
- Al subir `n_muestras`, la consistencia mejora (el voto se queda con los ítems estables y descarta los espurios) mientras la latencia crece más o menos en línea recta.
- Lo que me interesa para la tesis es el codo de la curva: el `n_muestras` a partir del cual la consistencia ya casi no mejora pero el coste sigue subiendo. Ese es el tamaño de lote que recomendaría.
